In [0]:
# Databricks notebook source
import sys as _sys
_nb = (dbutils.notebook.entry_point.getDbutils().notebook()
       .getContext().notebookPath().get())
_sys.path.insert(0, '/Workspace' + '/'.join(_nb.split('/')[:-2]) + '/src')
from lib.common import require_widget
from lib.domain_scanner import (
    scan_domain_assignments,
    suggest_domains,
    write_domain_violations,
)
dbutils.widgets.text("catalog",         "")
dbutils.widgets.text("control_schema",  "uc_hygiene")
dbutils.widgets.text("target_catalogs", "")
dbutils.widgets.text("dry_run",         "false")

catalog        = require_widget(dbutils, "catalog")
control_schema = require_widget(dbutils, "control_schema")
target_catalogs = [
    c.strip() for c in require_widget(dbutils, "target_catalogs").split(",") if c.strip()
]
dry_run = dbutils.widgets.get("dry_run").lower() == "true"

print(f"Control: {catalog}.{control_schema}")
print(f"Scanning: {target_catalogs}")
print(f"Dry run: {dry_run}")
import time as _t; _task_start = _t.time()


In [0]:
# Scan for tables missing domain assignment
unassigned = scan_domain_assignments(
    spark, catalog, control_schema, target_catalogs
)
unassigned_count = unassigned.count()
print(f"Tables without domain: {unassigned_count}")


In [0]:
# Suggest domains based on schema-level majority voting
suggestions = suggest_domains(spark, unassigned, catalog, control_schema)

suggested_count = suggestions.filter("suggested_domain IS NOT NULL").count()
print(f"Domain suggestions generated: {suggested_count} / {unassigned_count}")
if suggested_count > 0:
    suggestions.filter("suggested_domain IS NOT NULL").show(20, truncate=False)


In [0]:
# Write violations to scan_results (respects dry_run)
written = write_domain_violations(
    spark, suggestions, catalog, control_schema, dry_run=dry_run
)


In [0]:
_duration = int(_t.time() - _task_start)
print(f"\n{'='*52}")
print(f"  DOMAIN ASSIGNMENT SCAN COMPLETE")
print(f"{'='*52}")
print(f"  Tables scanned:   (all managed tables)")
print(f"  Unassigned:       {unassigned_count}")
print(f"  Suggestions made: {suggested_count}")
print(f"  Dry run:          {dry_run}")
print(f"  Duration:         {_duration}s")
print(f"{'='*52}")
